In [ ]:
import numpy as np
from numba import njit, prange, set_num_threads
from time import perf_counter_ns

def matrix_product(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in range(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

@njit(parallel=True)
def matrix_product_par(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""

    set_num_threads(2);

    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in prange(m):
        for col_ind in prange(n):
            for k in prange(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c

a = np.random.randn(100, 100)
b = np.random.randn(100, 100)

c_actual_par = matrix_product_par(a, b)
#segunda corrida
t_ini = perf_counter_ns()
c_actual_par = matrix_product_par(a, b)
t_fin = perf_counter_ns()
t_par = t_fin - t_ini

t_ini = perf_counter_ns()
c_actual = matrix_product(a, b)
t_fin = perf_counter_ns()
t_norm = t_fin - t_ini

t_ini = perf_counter_ns()
c_expected = a @ b
t_fin = perf_counter_ns()
t_np = t_fin - t_ini

error = np.linalg.norm(c_actual - c_expected) / np.linalg.norm(c_expected)
error_par = np.linalg.norm(c_actual_par - c_expected) / np.linalg.norm(c_expected)

print(f"The error is {error}, the parallel error is {error_par}.")
print(f"The times are: parallel = {t_par} -- manual = {t_norm} -- numpy = {t_np}")

The error is 2.3243142369270957e-16, the parallel error is 2.3243142369270957e-16.
The times are: parallel = 1068648 -- manual = 676512964 -- numpy = 5316491


In [ ]:
%%writefile omp_mat_pre.c

#include <omp.h>
#include <errno.h>
#include <limits.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static int obtener_cantidad(int filas, int columnas, size_t *cantidad)
{
    if (filas <= 0 || columnas <= 0 ||
        (size_t)filas > SIZE_MAX / (size_t)columnas) {
        return 0;
    }

    *cantidad = (size_t)filas * (size_t)columnas;
    return *cantidad <= SIZE_MAX / sizeof(double);
}

static void multiplicar_serial(const double *a, const double *b, double *c,
                               int filas_a, int columnas_a, int columnas_b)
{
    for (int i = 0; i < filas_a; ++i) {
        for (int j = 0; j < columnas_b; ++j) {
            double suma = 0.0;
            for (int k = 0; k < columnas_a; ++k) {
                suma += a[(size_t)i * columnas_a + k] *
                        b[(size_t)k * columnas_b + j];
            }
            c[(size_t)i * columnas_b + j] = suma;
        }
    }
}

static void multiplicar_openmp(const double *a, const double *b, double *c,
                               int filas_a, int columnas_a, int columnas_b)
{
    #pragma omp parallel for collapse(2) schedule(static)
    for (int i = 0; i < filas_a; ++i) {
        for (int j = 0; j < columnas_b; ++j) {
            double suma = 0.0;
            for (int k = 0; k < columnas_a; ++k) {
                suma += a[(size_t)i * columnas_a + k] *
                        b[(size_t)k * columnas_b + j];
            }
            c[(size_t)i * columnas_b + j] = suma;
        }
    }
}

static void imprimir_matriz(const char *nombre, const double *matriz,
                            int filas, int columnas)
{
    printf("%s:\n", nombre);
    for (int i = 0; i < filas; ++i) {
        for (int j = 0; j < columnas; ++j) {
            printf("%8.0f", matriz[(size_t)i * columnas + j]);
        }
        putchar('\n');
    }
}

int main(int argc, char *argv[])
{
    if (argc != 2) {
        fprintf(stderr, "Uso: %s N\n", argv[0]);
        return EXIT_FAILURE;
    }

    errno = 0;
    char *fin;
    long valor_n = strtol(argv[1], &fin, 10);
    if (errno != 0 || fin == argv[1] || *fin != '\0' ||
        valor_n <= 0 || valor_n > INT_MAX) {
        fprintf(stderr, "N debe ser un entero positivo.\n");
        return EXIT_FAILURE;
    }

    const int n = (int)valor_n;
    size_t cantidad;

    if (!obtener_cantidad(n, n, &cantidad)) {
        fprintf(stderr, "N es demasiado grande para reservar las matrices.\n");
        return EXIT_FAILURE;
    }

    double *a = malloc(cantidad * sizeof(*a));
    double *b = malloc(cantidad * sizeof(*b));
    double *c_serial = malloc(cantidad * sizeof(*c_serial));
    double *c_openmp = malloc(cantidad * sizeof(*c_openmp));
    if (a == NULL || b == NULL || c_serial == NULL || c_openmp == NULL) {
        fprintf(stderr, "No se pudo reservar memoria para las matrices.\n");
        free(a);
        free(b);
        free(c_serial);
        free(c_openmp);
        return EXIT_FAILURE;
    }

    srand((unsigned int)time(NULL));
    for (size_t i = 0; i < cantidad; ++i) {
        a[i] = rand() % 10;
        b[i] = rand() % 10;
    }

    printf("Multiplicando matrices aleatorias de %d x %d.\n", n, n);
    double inicio = omp_get_wtime();
    multiplicar_serial(a, b, c_serial, n, n, n);
    double tiempo_serial = (omp_get_wtime() - inicio) * 1000000000;

    multiplicar_openmp(a, b, c_openmp, n, n, n);
    inicio = omp_get_wtime();
    multiplicar_openmp(a, b, c_openmp, n, n, n);
    double tiempo_openmp = (omp_get_wtime() - inicio) * 1000000000;

    int resultados_iguales = 1;
    for (size_t i = 0; i < cantidad; ++i) {
        if (c_serial[i] != c_openmp[i]) {
            resultados_iguales = 0;
            break;
        }
    }

    if (n <= 10) {
        imprimir_matriz("A", a, n, n);
        imprimir_matriz("B", b, n, n);
        imprimir_matriz("Resultado", c_openmp, n, n);
    } else {
        printf("Matrices omitidas en pantalla porque N es mayor que 10.\n");
    }
    printf("Resultados iguales: %s\n", resultados_iguales ? "si" : "no");
    printf("Tiempo serial:  %.6f nanosegundos\n", tiempo_serial);
    printf("Tiempo OpenMP:  %.6f nanosegundos (%d hilos maximos)\n",
           tiempo_openmp, omp_get_max_threads());

    free(a);
    free(b);
    free(c_serial);
    free(c_openmp);
    return resultados_iguales ? EXIT_SUCCESS : EXIT_FAILURE;
}

Overwriting omp_mat_pre.c


In [ ]:
!gcc -o matrices -O3 -match=native omp_mat_pre.c -fopenmp
!OMP_NUM_THREADS=2 ./matrices 100

gcc: error: unrecognized command-line option ‘-match=native’; did you mean ‘-march=native’?
Multiplicando matrices aleatorias de 100 x 100.
Matrices omitidas en pantalla porque N es mayor que 10.
Resultados iguales: si
Tiempo serial:  3816232.999952 nanosegundos
Tiempo OpenMP:  3312822.999987 nanosegundos (2 hilos maximos)
